# Run Full Stack Local + LAN IP + MCP Overview

Este notebook ayuda a:

- arrancar el full stack local de Turbobujias
- calcular la IP local/LAN para pruebas en la red
- resumir los MCP servers definidos en `.vscode/mcp.json`
- distinguir entre uso local (`stdio`) y remoto (`http`)

Puedes usar `CHATBOT_MODE = "local"` o `CHATBOT_MODE = "remote"` antes de ejecutar la celda de arranque.

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_ROOT = Path(r"c:\Users\ASUS\tbai\Turbobujias-pro")
CHATBOT_DIR = REPO_ROOT / "turbobujias-ai"
BACKEND_DIR = REPO_ROOT / "backend"
FRONTEND_DIR = REPO_ROOT / "turbobujias-web"
NODE_CMD = shutil.which("npm") or "npm"
HOST_PYTHON_CMD = sys.executable
CHATBOT_VENV_DIR = CHATBOT_DIR / ".venv313"
CHATBOT_PYTHON = CHATBOT_VENV_DIR / "Scripts" / "python.exe"

if not CHATBOT_PYTHON.exists():
    subprocess.run([HOST_PYTHON_CMD, "-m", "venv", str(CHATBOT_VENV_DIR)], cwd=CHATBOT_DIR, check=True)

install_steps = [
    ("Upgrade host pip", [HOST_PYTHON_CMD, "-m", "pip", "install", "--upgrade", "pip"], REPO_ROOT),
    ("Upgrade chatbot venv pip", [str(CHATBOT_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], CHATBOT_DIR),
    ("Install chatbot Python requirements into .venv313", [str(CHATBOT_PYTHON), "-m", "pip", "install", "-r", str(CHATBOT_DIR / "requirements.txt")], CHATBOT_DIR),
    ("Install backend Node dependencies", [NODE_CMD, "install"], BACKEND_DIR),
    ("Install frontend Node dependencies", [NODE_CMD, "install"], FRONTEND_DIR),
]

results = []
for label, command, cwd in install_steps:
    print(f"[deps] running: {label}")
    completed = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )
    results.append({
        "step": label,
        "returncode": completed.returncode,
        "stdout_tail": completed.stdout[-500:] if completed.stdout else "",
        "stderr_tail": completed.stderr[-500:] if completed.stderr else "",
    })

results

[deps] running: Upgrade host pip


[deps] running: Upgrade chatbot venv pip
[deps] running: Install chatbot Python requirements into .venv313
[deps] running: Install backend Node dependencies
[deps] running: Install frontend Node dependencies


[{'step': 'Upgrade host pip',
  'returncode': 0,
  'stdout_tail': 'Requirement already satisfied: pip in c:\\Users\\ASUS\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (26.0.1)\n',
  'stderr_tail': ''},
 {'step': 'Upgrade chatbot venv pip',
  'returncode': 0,
  'stdout_tail': 'Requirement already satisfied: pip in .\\.venv313\\Lib\\site-packages (26.0.1)\n',
  'stderr_tail': ''},
 {'step': 'Install chatbot Python requirements into .venv313',
  'returncode': 0,
  'stdout_tail': 'SUS\\tbai\\Turbobujias-pro\\turbobujias-ai\\requirements.txt (line 3)) (3.6.0)\nRequirement already satisfied: mpmath<1.4,>=1.1.0 in .\\.venv313\\Lib\\site-packages (from sympy>=1.13.3->torch>=2.2.2->-r c:\\Users\\ASUS\\tbai\\Turbobujias-pro\\turbobujias-ai\\requirements.txt (line 12)) (1.3.0)\nRequirement already satisfied: llvmlite<0.48,>=0.47.0dev0 in .\\.venv313\\Lib\\site-packages (from numba->openai-whisper>=20231117->-r c:\\Users\\ASUS\\tbai\\Turbobujias-pro\\turbobujias-ai\\requirements

In [2]:
from pathlib import Path
import json
import os
import platform
import socket
import subprocess
import time
from urllib.request import urlopen

REPO_ROOT = Path(r"c:\Users\ASUS\tbai\Turbobujias-pro")
DEV_STACK_SCRIPT = REPO_ROOT / "scripts" / "dev-stack.mjs"
MCP_CONFIG_PATH = REPO_ROOT / ".vscode" / "mcp.json"
PID_FILE = REPO_ROOT / "dist" / "fullstack-local.pid"
LOG_FILE = REPO_ROOT / "dist" / "fullstack-local.log"
BACKEND_PORT = 3001
FRONTEND_PORT = 3000
CHATBOT_PORT = 7860
CHATBOT_MODE = "local"  # change to "remote" if you want the deployed Space instead of local Python chatbot
REMOTE_CHATBOT_URL = "https://sjhallo07-turbobujias-ai.hf.space"

print({
    "repo_root": str(REPO_ROOT),
    "dev_stack_script_exists": DEV_STACK_SCRIPT.exists(),
    "mcp_config_exists": MCP_CONFIG_PATH.exists(),
    "chatbot_mode": CHATBOT_MODE,
})

{'repo_root': 'c:\\Users\\ASUS\\tbai\\Turbobujias-pro', 'dev_stack_script_exists': True, 'mcp_config_exists': True, 'chatbot_mode': 'local'}


In [3]:
def detect_lan_ip():
    candidates = []
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
            s.connect(("8.8.8.8", 80))
            ip = s.getsockname()[0]
            if ip and not ip.startswith("127."):
                candidates.append(ip)
    except OSError:
        pass

    try:
        for info in socket.getaddrinfo(socket.gethostname(), None, socket.AF_INET):
            ip = info[4][0]
            if ip and not ip.startswith("127.") and ip not in candidates:
                candidates.append(ip)
    except OSError:
        pass

    return candidates[0] if candidates else "127.0.0.1"

LAN_IP = detect_lan_ip()
urls = {
    "frontend_local": f"http://127.0.0.1:{FRONTEND_PORT}",
    "frontend_lan": f"http://{LAN_IP}:{FRONTEND_PORT}",
    "backend_local_health": f"http://127.0.0.1:{BACKEND_PORT}/api/health",
    "backend_lan_health": f"http://{LAN_IP}:{BACKEND_PORT}/api/health",
    "chatbot_local": f"http://127.0.0.1:{CHATBOT_PORT}",
    "chatbot_lan": f"http://{LAN_IP}:{CHATBOT_PORT}",
    "chatbot_remote": REMOTE_CHATBOT_URL,
}
urls

{'frontend_local': 'http://127.0.0.1:3000',
 'frontend_lan': 'http://192.168.0.12:3000',
 'backend_local_health': 'http://127.0.0.1:3001/api/health',
 'backend_lan_health': 'http://192.168.0.12:3001/api/health',
 'chatbot_local': 'http://127.0.0.1:7860',
 'chatbot_lan': 'http://192.168.0.12:7860',
 'chatbot_remote': 'https://sjhallo07-turbobujias-ai.hf.space'}

In [4]:
mcp_config = json.loads(MCP_CONFIG_PATH.read_text(encoding="utf-8"))
server_summary = []
for name, config in mcp_config.get("servers", {}).items():
    server_summary.append({
        "name": name,
        "type": config.get("type", "unknown"),
        "command_or_url": config.get("command") or config.get("url"),
        "mode": "remote" if config.get("type") == "http" else "local/studio",
    })
server_summary

[{'name': 'hf-mcp-server',
  'type': 'http',
  'command_or_url': 'https://huggingface.co/mcp?login&mix=proxy',
  'mode': 'remote'},
 {'name': 'hf-mcp-server-bouquet',
  'type': 'http',
  'command_or_url': 'https://huggingface.co/mcp?login&bouquet=proxy',
  'mode': 'remote'},
 {'name': 'io.github.github/github-mcp-server',
  'type': 'stdio',
  'command_or_url': 'docker',
  'mode': 'local/studio'},
 {'name': 'my-mcp-server-607bb799',
  'type': 'stdio',
  'command_or_url': 'deploy',
  'mode': 'local/studio'}]

In [6]:
def _pid_is_running(pid_text):
    try:
        pid_value = int(str(pid_text).strip())
    except (TypeError, ValueError):
        return False
    try:
        if platform.system() == "Windows":
            completed = subprocess.run(
                ["tasklist", "/FI", f"PID eq {pid_value}"],
                capture_output=True,
                text=True,
                check=False,
            )
            return str(pid_value) in completed.stdout
        os.kill(pid_value, 0)
        return True
    except Exception:
        return False

def start_full_stack(chatbot_mode=CHATBOT_MODE, scope="full"):
    PID_FILE.parent.mkdir(parents=True, exist_ok=True)
    if PID_FILE.exists():
        existing = PID_FILE.read_text(encoding="utf-8").strip()
        if _pid_is_running(existing):
            return {"status": "already-recorded", "pid": existing, "pid_file": str(PID_FILE)}
        PID_FILE.unlink(missing_ok=True)

    if scope == "chatbot-only":
        command = [str(CHATBOT_PYTHON), "app.py"]
        target_cwd = CHATBOT_DIR
        env = os.environ.copy()
        env["PORT"] = str(CHATBOT_PORT)
        env["GRADIO_SERVER_NAME"] = "127.0.0.1"
    else:
        command = ["node", str(DEV_STACK_SCRIPT), "--chatbot", chatbot_mode]
        target_cwd = REPO_ROOT
        env = None

    with open(LOG_FILE, "a", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=target_cwd,
            env=env,
            stdout=log_file,
            stderr=log_file,
            stdin=subprocess.DEVNULL,
            creationflags=subprocess.CREATE_NEW_PROCESS_GROUP if platform.system() == "Windows" else 0,
        )
    PID_FILE.write_text(str(process.pid), encoding="utf-8")
    time.sleep(5)
    return {
        "status": "started",
        "pid": process.pid,
        "scope": scope,
        "chatbot_mode": chatbot_mode,
        "log_file": str(LOG_FILE),
        "frontend": urls["frontend_local"] if scope == "full" else None,
        "backend_health": urls["backend_local_health"] if scope == "full" else None,
        "chatbot": urls["chatbot_local"] if chatbot_mode == "local" else urls["chatbot_remote"],
    }

START_SCOPE = "chatbot-only"
last_start = start_full_stack(scope=START_SCOPE)
last_start

{'status': 'started',
 'pid': 8580,
 'scope': 'chatbot-only',
 'chatbot_mode': 'local',
 'log_file': 'c:\\Users\\ASUS\\tbai\\Turbobujias-pro\\dist\\fullstack-local.log',
 'frontend': None,
 'backend_health': None,
 'chatbot': 'http://127.0.0.1:7860'}

In [7]:
def try_fetch(url):
    try:
        with urlopen(url, timeout=5) as response:
            return {"url": url, "status": response.status}
    except Exception as exc:
        return {"url": url, "status": "error", "detail": str(exc)}

targets = []
if globals().get("START_SCOPE") == "chatbot-only":
    targets.append(urls["chatbot_local"] if CHATBOT_MODE == "local" else urls["chatbot_remote"])
else:
    targets.extend([
        urls["frontend_local"],
        urls["backend_local_health"],
        urls["chatbot_local"] if CHATBOT_MODE == "local" else urls["chatbot_remote"],
    ])

health_checks = [try_fetch(url) for url in targets]
health_checks

[{'url': 'http://127.0.0.1:7860',
  'status': 'error',
  'detail': '<urlopen error [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión>'}]

In [7]:
def stop_full_stack():
    if not PID_FILE.exists():
        return {"status": "not-running", "pid_file": str(PID_FILE)}

    pid = PID_FILE.read_text(encoding="utf-8").strip()
    if platform.system() == "Windows":
        subprocess.run(["taskkill", "/PID", pid, "/T", "/F"], check=False, capture_output=True, text=True)
    else:
        subprocess.run(["kill", "-TERM", pid], check=False, capture_output=True, text=True)

    PID_FILE.unlink(missing_ok=True)
    return {"status": "stopped", "pid": pid, "log_file": str(LOG_FILE)}

# Run this cell only when you want to stop the local stack.
# stop_full_stack()

## Uso práctico

- **Modo local completo:** deja `CHATBOT_MODE = "local"` y ejecuta las celdas 2 a 6.
- **Modo híbrido:** cambia `CHATBOT_MODE = "remote"` para usar el Space desplegado del chatbot mientras el frontend y backend corren localmente.
- **MCP local/studio:** en `.vscode/mcp.json`, los servidores `stdio` son para uso local o en VS Code Studio.
- **MCP remoto:** en `.vscode/mcp.json`, los servidores `http` son remotos y requieren token/configuración.
- **Pruebas en la red local:** usa las URLs `frontend_lan`, `backend_lan_health` y `chatbot_lan` mostradas por la celda 3.